# Verificación de los documentos de un postulante

Compara cada documento que entregó el postulante (`postulant_files/<Nombre>/`: PDF nativo, PDF escaneado o fotos) con su dato oficial en JSON (`webscrap_files/<Nombre>/`) y emite un dictamen por documento y la aptitud para el cargo.

**Antes de empezar**
- Abra el notebook desde la carpeta raíz del proyecto, con un kernel que tenga instalado `requirements.txt`.
- Configure la celda de configuración y ejecute todas las celdas en orden (*Run All*).
- Los PDF nativos se leen directo (sin OCR); las fotos y escaneos pasan por el OCR en la GPU (máx. 2,5 GB de VRAM).


## 1. Configuración

- `APPLICANT`: nombre de la carpeta del postulante dentro de `APPLICANTS_DIR`.
- `APPLICANTS_DIR` / `OFFICIAL_DIR`: carpetas con los documentos entregados y con los JSON oficiales (una subcarpeta por postulante, con el mismo nombre).
- `TODAY`: fecha de referencia para calcular vigencias.
- `USE_GPU`: `False` para leer con CPU (más lento).


In [ ]:
from datetime import date
from pathlib import Path

ROOT = Path.cwd()
APPLICANT = "NOMBRE APELLIDO APELLIDO"
APPLICANTS_DIR = ROOT / "postulant_files"
OFFICIAL_DIR = ROOT / "webscrap_files"
TODAY = date.today()
USE_GPU = True

## 2. Verificación

El número de identidad declarado y el cargo se toman del registro de postulantes (`postulant_files/candidatos.xlsx`) si el postulante está registrado.


In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from verification import config as C
from verification import pipeline as PL

registry = PL.read_registry(C.APPLICANT_REGISTRY)
people, documents, rejected_files, meta = PL.verify([APPLICANT], registry, APPLICANTS_DIR, OFFICIAL_DIR,
                                                    workers=4, use_gpu=USE_GPU, today=TODAY)
p = people[0]
print(f"{p['carpeta']} | identidad {p['dni'] or '(no registrado)'} | cargo: {p['puesto']}")
print(f"Autenticidad: {p['veredicto']} | Aptitud: {p['aptitud']}")
for m in p['motivos_aptitud']:
    print('  -', m)

## 3. Dictamen por documento


In [ ]:
for d in documents:
    print(f"
{d['archivo']}
  {d['tipo_documento']} | {d['formato']} | {d['veredicto']} | dato oficial: {d['archivo_oficial'] or '-'}")
    for m in d['motivos']:
        print('   •', m)
for x in rejected_files:
    print(f"
NO ADMITIDO: {x['archivo']} ({x['motivo']})")

## 4. Campo por campo

Solo los campos que no coinciden exactamente con el dato oficial.


In [ ]:
import pandas as pd
rows = [{"archivo": d['archivo'][:40], "campo": c['zona'], "oficial (JSON)": c['oficial'], "leído": c['postulante'],
         "estado": c['estado'], "motivo": c['motivo'], "confianza": c['conf']}
        for d in documents for c in d['campos'] if c['estado'] not in ('IGUAL', 'NO_APLICA')]
pd.DataFrame(rows)